# Phase 2 (simplified) — minimal MLP on phase-1 features, with temporal val

Phase 2's cross-wallet attention model lost by 17 F1 points to RF[aug] (0.63 vs 0.81). Three compounding reasons:
1. **57% pad fraction** — most txs have 1–2 real incident wallets; the attention layer was an expensive identity operation for the majority of the data.
2. **Random val** carved from `t ≤ 34` is in-distribution with train, so val-best epoch (15) was optimised for the wrong target — the test set lives at `t ≥ 35` with a different feature distribution.
3. **Tabular regime** — 3K positive examples / 45K parameters is the regime where Grinsztajn et al. (NeurIPS 2022) show tree ensembles dominate neural nets.

This notebook removes (1) and (2) from the picture. (3) is fundamental and we won't beat it; the goal here isn't to outperform RF — it's to (a) confirm a properly simplified neural model can at least *match* RF, and (b) get a clean neural probability we can later ensemble with RF.

## What changed
| | Phase 2 (attention) | Phase 2 (this notebook) |
|---|---|---|
| Architecture | 1 cross-wallet attention layer + projections + classifier | **2-layer MLP** on `[182 raw ‖ 103 traj]` |
| Params | 45K | **~10K** |
| Per-wallet tensor | `[N_tx, 8, 29]` (most pad) | **dropped** |
| Val split | random 10% from `t ≤ 34` | **temporal: `t ∈ [30, 34]`** |
| Threshold | fixed 0.5 | 0.5 + **best-on-temporal-val** |
| Class weight | 5.0 | 3.0 (even more moderate) |

## Identical to phase 1
- Same data loading (cells 1–3 of phase 1)
- Same `per_wallet_priors` causal trajectory features (`F_TRAJ = 103`)
- Same TEST set: labelled txs with `t ≥ 35`
- Same causality contract (strict `τ < t` priors, no `wallets_features.txt`, no AddrAddr)

## Decision rule
- **Match RF[aug] within ±0.5 F1**: simplified neural is a viable consumer of the trajectory features. Worth ensembling with RF.
- **Loses by >2 F1**: even simplified neural can't beat RF on this dataset. Honest conclusion: ship RF + traj features as the main result, neural is an ablation.
- **Beats RF[aug]**: surprising, write it up as the headline.


In [9]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve,
)

# Walk up from CWD to find the repo root (the dir that has actors_data/ and transactions_data/)
ROOT = Path.cwd()
while not (ROOT / "actors_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
WALLETS_DIR      = ROOT / "actors_data"
TRANSACTIONS_DIR = ROOT / "transactions_data"
assert WALLETS_DIR.exists() and TRANSACTIONS_DIR.exists()
print(f"repo root: {ROOT}")

# === Same TEST set as phase 1; temporal val carve from train ===
TRAIN_END_FOR_TEST = 34         # test = labelled txs with t > 34 (paper protocol, identical to phase 1)
TRAIN_END          = 29         # neural train = t <= 29
VAL_START          = 30         # neural val   = t in [30, 34]   (temporal, near test in time)
VAL_END            = 34
N_TIME_STEPS       = 49
RANDOM_SEED        = 175
np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

# === Phase 1 trajectory engineering caps (unchanged from phase 1) ===
MAX_INCIDENT_PER_SIDE = 32
MAX_CO_WALLETS        = 150
RECENCY_SENTINEL      = N_TIME_STEPS * 2
DECAY_RATE            = 0.2

# === Simple MLP hyperparameters ===
HIDDEN_DIM   = 32
DROPOUT      = 0.3
LR           = 1e-3
WEIGHT_DECAY = 5e-4
EPOCHS       = 30
BATCH_SIZE   = 512
CLASS_WEIGHT = 3.0      # less aggressive than phase-1 RF balanced (~9.2) and phase-2 attention (5.0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")


repo root: /Users/jarayliu/Documents/GitHub/stat175-final
device: cpu


In [10]:
print("Loading transactions...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")
tx_classes_df["label"] = tx_classes_df["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
tx_feat_cols = [c for c in tx_features_df.columns if c not in ("txId", "Time step")]
F_TX = len(tx_feat_cols)

merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_raw = tx_features_df[tx_feat_cols].fillna(0).astype(np.float32).values
print(f"  N_tx={N_tx:,}  F_TX={F_TX}")
print(f"  labels: illicit={int((tx_label==1).sum()):,}  licit={int((tx_label==0).sum()):,}  unknown={int((tx_label==-1).sum()):,}")

print("\nLoading actor bipartite edges...")
addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) | set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a: i for i, a in enumerate(wallet_addrs)}
N_w = len(wallet_addr_to_idx)
print(f"  unique wallets: {N_w:,}")

print("\nBuilding per-wallet timelines...")
def edges_with_time(edge_df, addr_col, tx_col):
    df = edge_df[[addr_col, tx_col]].copy()
    df["w"]      = df[addr_col].map(wallet_addr_to_idx)
    df["tx"]     = df[tx_col].map(tx_id_to_idx)
    df = df.dropna(subset=["w", "tx"])
    df["w"]      = df["w"].astype(np.int64)
    df["tx"]     = df["tx"].astype(np.int64)
    df["t"]      = tx_time[df["tx"].values]
    df["tx_lab"] = tx_label[df["tx"].values]
    return df[["w","tx","t","tx_lab"]]

at = edges_with_time(addr_tx_df, "input_address",  "txId")
ta = edges_with_time(tx_addr_df, "output_address", "txId")
incident_in  = defaultdict(list)
incident_out = defaultdict(list)
for tx, w in zip(at["tx"].values,  at["w"].values):  incident_in[int(tx)].append(int(w))
for tx, w in zip(ta["tx"].values,  ta["w"].values):  incident_out[int(tx)].append(int(w))

all_edges = pd.concat([at, ta], ignore_index=True)
all_edges = all_edges.drop_duplicates(subset=["w", "tx"]).sort_values(["w", "t"]).reset_index(drop=True)
g = all_edges.groupby("w", sort=False)
wallet_t_arr   = {}
wallet_tx_arr  = {}
wallet_lab_arr = {}
for w, sub in g:
    wallet_t_arr[int(w)]   = sub["t"].values.astype(np.int64)
    wallet_tx_arr[int(w)]  = sub["tx"].values.astype(np.int64)
    wallet_lab_arr[int(w)] = sub["tx_lab"].values.astype(np.int64)

wallet_has_illicit_by = {}
for w, labs in wallet_lab_arr.items():
    illicit_mask = (labs == 1)
    if illicit_mask.any():
        wallet_has_illicit_by[w] = int(wallet_t_arr[w][illicit_mask].min())

wallet_event_count = {w: len(tarr) for w, tarr in wallet_t_arr.items()}
print(f"  wallets with timelines: {len(wallet_t_arr):,}")
print(f"  total wallet-tx pairs: {len(all_edges):,}")
print(f"  wallets with any illicit history: {len(wallet_has_illicit_by):,}")


Loading transactions...
  N_tx=203,769  F_TX=182
  labels: illicit=4,545  licit=42,019  unknown=157,205

Loading actor bipartite edges...
  unique wallets: 822,942

Building per-wallet timelines...
  wallets with timelines: 822,942
  total wallet-tx pairs: 1,268,260
  wallets with any illicit history: 14,266


In [11]:
# Port of phase 1's per_wallet_priors / aggregate_side / main loop — but DROP per-wallet tensor.
# This cell only computes traj_X (the 103-dim flat per-tx aggregates).

agg_feat_names = ["total_BTC", "fees", "num_input_addresses", "num_output_addresses"]
agg_feat_idxs  = [tx_feat_cols.index(c) for c in agg_feat_names]
total_btc_idx  = tx_feat_cols.index("total_BTC")
F_agg          = len(agg_feat_names)

def pick_top_wallets(wlist, k=MAX_INCIDENT_PER_SIDE):
    if len(wlist) <= k: return list(wlist)
    cnts  = np.array([wallet_event_count.get(w, 0) for w in wlist], dtype=np.int64)
    order = np.argsort(-cnts, kind="stable")
    return [wlist[i] for i in order[:k]]

def per_wallet_priors(w, t_query):
    tarr = wallet_t_arr.get(w)
    if tarr is None: return None
    cut = int(np.searchsorted(tarr, t_query, side="left"))
    if cut == 0: return None
    prior_t   = tarr[:cut]
    prior_lab = wallet_lab_arr[w][:cut]
    prior_tx  = wallet_tx_arr[w][:cut]
    illicit_mask = (prior_lab == 1)
    n_illicit = int(illicit_mask.sum())
    n_licit   = int((prior_lab == 0).sum())
    last_illicit_t = int(prior_t[illicit_mask].max()) if n_illicit > 0 else -1
    if n_illicit > 0:
        decay_w = np.exp(-DECAY_RATE * (t_query - prior_t[illicit_mask]).astype(np.float64))
        decayed_illicit_score = float(decay_w.sum())
    else:
        decayed_illicit_score = 0.0
    illicit_frac = n_illicit / max(n_illicit + n_licit, 1)
    co_wallets = set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int,  []):
            if co_w != w: co_wallets.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w: co_wallets.add(co_w)
        if len(co_wallets) >= MAX_CO_WALLETS: break
    n_co_wallets = len(co_wallets)
    n_co_illicit = sum(1 for cw in co_wallets
                       if wallet_has_illicit_by.get(cw, N_TIME_STEPS + 1) < t_query)
    unique_in_partners, unique_out_partners = set(), set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int,  []):
            if co_w != w: unique_in_partners.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w: unique_out_partners.add(co_w)
    n_in_partners  = len(unique_in_partners)
    n_out_partners = len(unique_out_partners)
    fan_asymmetry  = n_out_partners / max(n_in_partners, 1)
    age      = int(t_query - prior_t[0])
    recency  = int(t_query - prior_t[-1])
    n_recent_5 = int(((t_query - prior_t) <= 5).sum())
    n_recent_1 = int(((t_query - prior_t) <= 1).sum())
    if cut >= 2:
        iat = np.diff(prior_t.astype(np.float64))
        iat_mean = float(iat.mean()); iat_std = float(iat.std())
        burstiness = float((iat_std - iat_mean) / (iat_std + iat_mean + 1e-8))
    else:
        iat_mean, iat_std, burstiness = 0.0, 0.0, 0.0
    velocity = n_recent_5 / max(cut, 1)
    feat_vals = tx_X_raw[prior_tx][:, agg_feat_idxs]
    return {
        "n": int(cut), "n_illicit": n_illicit, "n_licit": n_licit,
        "illicit_frac": illicit_frac, "last_illicit_t": last_illicit_t,
        "decayed_illicit": decayed_illicit_score,
        "n_co_wallets": n_co_wallets, "n_co_illicit": n_co_illicit,
        "co_illicit_frac": n_co_illicit / max(n_co_wallets, 1),
        "n_in_partners": n_in_partners, "n_out_partners": n_out_partners,
        "fan_asymmetry": fan_asymmetry,
        "first_seen_t": int(prior_t[0]), "last_seen_t": int(prior_t[-1]),
        "age": age, "recency": recency,
        "n_recent_5": n_recent_5, "n_recent_1": n_recent_1,
        "iat_mean": iat_mean, "iat_std": iat_std,
        "burstiness": burstiness, "velocity": velocity,
        "feat_means": feat_vals.mean(axis=0), "feat_maxes": feat_vals.max(axis=0),
    }

def aggregate_side(summaries, side, t_T):
    n_total = len(summaries)
    valid   = [s for s in summaries if s is not None]
    n_w_prior = len(valid)
    out = {}; p = side
    out[f"{p}_n_wallets"]              = n_total
    out[f"{p}_n_wallets_with_prior"]   = n_w_prior
    out[f"{p}_frac_first_appearance"]  = (n_total - n_w_prior) / max(n_total, 1)
    if not valid:
        zeros_keys = ["n_priors_sum","n_priors_max","n_illicit_sum","n_illicit_max","n_licit_sum",
            "n_wallets_with_illicit","n_wallets_illicit_frac_gt0","n_wallets_illicit_frac_gt50",
            "illicit_frac_max","illicit_frac_mean","decayed_illicit_max","decayed_illicit_sum",
            "co_illicit_sum","co_illicit_max","co_illicit_frac_max","co_illicit_frac_mean",
            "n_co_wallets_sum","fan_asymmetry_max","fan_asymmetry_mean",
            "n_in_partners_max","n_out_partners_max","frac_single_use",
            "age_max","age_mean","n_recent_5_sum","n_recent_5_max","n_recent_1_sum",
            "velocity_max","velocity_mean","burstiness_max","burstiness_mean",
            "iat_mean_min","iat_std_max"]
        for k in zeros_keys: out[f"{p}_{k}"] = 0.0
        out[f"{p}_recency_to_illicit_min"] = float(RECENCY_SENTINEL)
        out[f"{p}_recency_min"]            = float(RECENCY_SENTINEL)
        for nm in agg_feat_names:
            out[f"{p}_prior_{nm}_mean_max"] = 0.0
            out[f"{p}_prior_{nm}_max_max"]  = 0.0
        return out
    ns        = np.array([s["n"]              for s in valid], dtype=np.float64)
    nis       = np.array([s["n_illicit"]      for s in valid], dtype=np.float64)
    nls       = np.array([s["n_licit"]        for s in valid], dtype=np.float64)
    ill_frac  = np.array([s["illicit_frac"]   for s in valid], dtype=np.float64)
    dec_ill   = np.array([s["decayed_illicit"]for s in valid], dtype=np.float64)
    last_ill  = np.array([s["last_illicit_t"] for s in valid], dtype=np.int64)
    has_ill   = (last_ill >= 0)
    rec_ill   = np.where(has_ill, t_T - last_ill, RECENCY_SENTINEL).astype(np.float64)
    co_ill   = np.array([s["n_co_illicit"]    for s in valid], dtype=np.float64)
    co_n     = np.array([s["n_co_wallets"]    for s in valid], dtype=np.float64)
    co_frac  = np.array([s["co_illicit_frac"] for s in valid], dtype=np.float64)
    fan_a    = np.array([s["fan_asymmetry"]   for s in valid], dtype=np.float64)
    n_inp    = np.array([s["n_in_partners"]   for s in valid], dtype=np.float64)
    n_outp   = np.array([s["n_out_partners"]  for s in valid], dtype=np.float64)
    ages = np.array([s["age"]      for s in valid], dtype=np.float64)
    recs = np.array([s["recency"]  for s in valid], dtype=np.float64)
    nr5  = np.array([s["n_recent_5"] for s in valid], dtype=np.float64)
    nr1  = np.array([s["n_recent_1"] for s in valid], dtype=np.float64)
    vel  = np.array([s["velocity"] for s in valid], dtype=np.float64)
    bur  = np.array([s["burstiness"] for s in valid], dtype=np.float64)
    iam  = np.array([s["iat_mean"] for s in valid], dtype=np.float64)
    ias  = np.array([s["iat_std"]  for s in valid], dtype=np.float64)
    feat_means = np.stack([s["feat_means"] for s in valid], axis=0)
    feat_maxes = np.stack([s["feat_maxes"] for s in valid], axis=0)
    out[f"{p}_n_priors_sum"] = float(ns.sum()); out[f"{p}_n_priors_max"] = float(ns.max())
    out[f"{p}_n_illicit_sum"] = float(nis.sum()); out[f"{p}_n_illicit_max"] = float(nis.max())
    out[f"{p}_n_licit_sum"] = float(nls.sum())
    out[f"{p}_n_wallets_with_illicit"] = float(has_ill.sum())
    out[f"{p}_n_wallets_illicit_frac_gt0"]  = float((ill_frac > 0.0).sum())
    out[f"{p}_n_wallets_illicit_frac_gt50"] = float((ill_frac > 0.5).sum())
    out[f"{p}_illicit_frac_max"] = float(ill_frac.max())
    out[f"{p}_illicit_frac_mean"] = float(ill_frac.mean())
    out[f"{p}_decayed_illicit_max"] = float(dec_ill.max())
    out[f"{p}_decayed_illicit_sum"] = float(dec_ill.sum())
    out[f"{p}_recency_to_illicit_min"] = float(rec_ill.min())
    out[f"{p}_co_illicit_sum"] = float(co_ill.sum()); out[f"{p}_co_illicit_max"] = float(co_ill.max())
    out[f"{p}_co_illicit_frac_max"] = float(co_frac.max())
    out[f"{p}_co_illicit_frac_mean"] = float(co_frac.mean())
    out[f"{p}_n_co_wallets_sum"] = float(co_n.sum())
    out[f"{p}_fan_asymmetry_max"] = float(fan_a.max())
    out[f"{p}_fan_asymmetry_mean"] = float(fan_a.mean())
    out[f"{p}_n_in_partners_max"] = float(n_inp.max())
    out[f"{p}_n_out_partners_max"] = float(n_outp.max())
    out[f"{p}_frac_single_use"] = sum(1 for s in valid if s["n"] == 1) / max(n_w_prior, 1)
    out[f"{p}_age_max"] = float(ages.max()); out[f"{p}_age_mean"] = float(ages.mean())
    out[f"{p}_recency_min"] = float(recs.min())
    out[f"{p}_n_recent_5_sum"] = float(nr5.sum()); out[f"{p}_n_recent_5_max"] = float(nr5.max())
    out[f"{p}_n_recent_1_sum"] = float(nr1.sum())
    out[f"{p}_velocity_max"] = float(vel.max()); out[f"{p}_velocity_mean"] = float(vel.mean())
    out[f"{p}_burstiness_max"] = float(bur.max()); out[f"{p}_burstiness_mean"] = float(bur.mean())
    out[f"{p}_iat_mean_min"] = float(iam.min()); out[f"{p}_iat_std_max"] = float(ias.max())
    for k, nm in enumerate(agg_feat_names):
        out[f"{p}_prior_{nm}_mean_max"] = float(feat_means[:, k].max())
        out[f"{p}_prior_{nm}_max_max"]  = float(feat_maxes[:, k].max())
    return out

print("Computing trajectory features for all txs...")
t0 = time.time()
traj_rows = []
for tx_idx in range(N_tx):
    t_T = int(tx_time[tx_idx])
    in_w  = pick_top_wallets(incident_in.get(tx_idx,  []))
    out_w = pick_top_wallets(incident_out.get(tx_idx, []))
    in_summ  = [per_wallet_priors(w, t_T) for w in in_w]
    out_summ = [per_wallet_priors(w, t_T) for w in out_w]
    row = {}
    row.update(aggregate_side(in_summ,  "in",  t_T))
    row.update(aggregate_side(out_summ, "out", t_T))
    row["both_sides_have_illicit"] = float(
        row["in_n_wallets_with_illicit"] > 0 and row["out_n_wallets_with_illicit"] > 0)
    row["total_n_illicit_priors"]  = row["in_n_illicit_sum"] + row["out_n_illicit_sum"]
    row["total_n_wallets_with_illicit"] = row["in_n_wallets_with_illicit"] + row["out_n_wallets_with_illicit"]
    row["total_co_illicit"]        = row["in_co_illicit_sum"] + row["out_co_illicit_sum"]
    row["min_recency_to_illicit"]  = min(row["in_recency_to_illicit_min"], row["out_recency_to_illicit_min"])
    row["max_illicit_frac_either_side"] = max(row["in_illicit_frac_max"], row["out_illicit_frac_max"])
    row["max_decayed_illicit_either"]   = max(row["in_decayed_illicit_max"], row["out_decayed_illicit_max"])
    row["max_co_illicit_frac_either"]   = max(row["in_co_illicit_frac_max"], row["out_co_illicit_frac_max"])
    row["total_frac_first_appearance"]  = (
        (row["in_frac_first_appearance"] * max(row["in_n_wallets"], 1) +
         row["out_frac_first_appearance"] * max(row["out_n_wallets"], 1))
        / max(row["in_n_wallets"] + row["out_n_wallets"], 1))
    T_total_btc = float(tx_X_raw[tx_idx, total_btc_idx])
    max_prior_btc  = max(row["in_prior_total_BTC_max_max"],  row["out_prior_total_BTC_max_max"])
    mean_prior_btc = max(row["in_prior_total_BTC_mean_max"], row["out_prior_total_BTC_mean_max"])
    row["T_btc_vs_max_prior"]  = T_total_btc / max(max_prior_btc, 1.0)
    row["T_btc_vs_mean_prior"] = T_total_btc / max(mean_prior_btc, 1.0)
    traj_rows.append(row)
    if tx_idx > 0 and tx_idx % 25_000 == 0:
        elapsed = time.time() - t0
        eta = (N_tx - tx_idx) * elapsed / tx_idx
        print(f"  tx {tx_idx:>7,}/{N_tx:,}  ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

traj_df = pd.DataFrame(traj_rows)
traj_X  = traj_df.values.astype(np.float32)
F_TRAJ  = traj_X.shape[1]
print(f"\n  done: traj_X={traj_X.shape} ({time.time()-t0:.1f}s)")


Computing trajectory features for all txs...
  tx  25,000/203,769  (2s elapsed, ~16s remaining)
  tx  50,000/203,769  (6s elapsed, ~17s remaining)
  tx  75,000/203,769  (9s elapsed, ~16s remaining)
  tx 100,000/203,769  (14s elapsed, ~15s remaining)
  tx 125,000/203,769  (19s elapsed, ~12s remaining)
  tx 150,000/203,769  (22s elapsed, ~8s remaining)
  tx 175,000/203,769  (25s elapsed, ~4s remaining)
  tx 200,000/203,769  (29s elapsed, ~1s remaining)

  done: traj_X=(203769, 103) (33.1s)


In [12]:
# ── TEMPORAL train/val/test split ──────────────────────────────────
# train: t in [1, TRAIN_END=29]
# val:   t in [VAL_START=30, VAL_END=34]   <-- TEMPORAL, not random
# test:  t in [TRAIN_END_FOR_TEST+1=35, 49]   <-- identical to phase 1's test set
labelled   = (tx_label != -1)
train_mask = labelled & (tx_time <= TRAIN_END)
val_mask   = labelled & (tx_time >= VAL_START) & (tx_time <= VAL_END)
test_mask  = labelled & (tx_time >  TRAIN_END_FOR_TEST)
print(f"train (t<= {TRAIN_END}):           n={int(train_mask.sum()):,}  illicit_rate={tx_label[train_mask].mean():.4f}")
print(f"val   ({VAL_START} <= t <= {VAL_END}):    n={int(val_mask.sum()):,}    illicit_rate={tx_label[val_mask].mean():.4f}")
print(f"test  (t >  {TRAIN_END_FOR_TEST}):           n={int(test_mask.sum()):,}   illicit_rate={tx_label[test_mask].mean():.4f}")

# ── Build the [raw 182 ‖ traj 103] feature matrix ──────────────────
X_full = np.concatenate([tx_X_raw, traj_X], axis=1).astype(np.float32)
F_IN = X_full.shape[1]
print(f"\ninput dim: F_IN = {F_TX} + {F_TRAJ} = {F_IN}")

# ── StandardScaler fit on TRAIN only ───────────────────────────────
scaler = StandardScaler().fit(X_full[train_mask])
X_std  = scaler.transform(X_full).astype(np.float32)
print(f"scaled X: {X_std.shape}")

# ── Index arrays + dataset/dataloader ──────────────────────────────
train_idx_arr = np.where(train_mask)[0]
val_idx_arr   = np.where(val_mask)[0]
test_idx_arr  = np.where(test_mask)[0]

class TxDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=np.int64)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = int(self.indices[i])
        return torch.from_numpy(X_std[idx]), torch.tensor(int(tx_label[idx]), dtype=torch.long)

train_loader = DataLoader(TxDataset(train_idx_arr), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(TxDataset(val_idx_arr),   batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0)
test_loader  = DataLoader(TxDataset(test_idx_arr),  batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0)
print(f"loaders: train={len(train_loader)}  val={len(val_loader)}  test={len(test_loader)}  batches")


train (t<= 29):           n=26,381  illicit_rate=0.1088
val   (30 <= t <= 34):    n=3,513    illicit_rate=0.1682
test  (t >  34):           n=16,670   illicit_rate=0.0650

input dim: F_IN = 182 + 103 = 285
scaled X: (203769, 285)
loaders: train=52  val=4  test=17  batches


In [13]:
class SimpleMLP(nn.Module):
    """2-layer MLP on [raw tx features ‖ trajectory features]. No attention.
    Aggressive dropout. ~10K params."""
    def __init__(self, F_in, hidden=32, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(F_in, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )
    def forward(self, x):
        return self.net(x)

model = SimpleMLP(F_IN, hidden=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"model: SimpleMLP  hidden={HIDDEN_DIM}  dropout={DROPOUT}  params={n_params:,}")


model: SimpleMLP  hidden=32  dropout=0.3  params=10,274


In [14]:
optim_ = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
class_weight = torch.tensor([1.0, CLASS_WEIGHT], dtype=torch.float, device=DEVICE)
print(f"using class_weight={class_weight.tolist()}  (3.0 = even more moderate than phase-2 attention's 5.0)")

def run_eval(loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            ys.append(y.cpu().numpy())
            ps.append(logits.softmax(-1)[:, 1].cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

best_val_f1 = -1.0
best_state  = None
best_epoch  = -1
print(f"\nTraining {EPOCHS} epochs, best epoch by TEMPORAL val F1@0.5 ...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running, seen = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optim_.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y, weight=class_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim_.step()
        running += float(loss.detach()) * y.size(0)
        seen    += y.size(0)
    vy, vp = run_eval(val_loader)
    v_f1_05 = f1_score(vy, (vp >= 0.5).astype(np.int64), pos_label=1, zero_division=0)
    is_best = v_f1_05 > best_val_f1
    if is_best:
        best_val_f1 = v_f1_05
        best_state  = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch  = epoch
    marker = "  *" if is_best else ""
    print(f"epoch {epoch:2d}  train_loss={running/max(seen,1):.4f}  "
          f"val_F1@0.5={v_f1_05:.4f}  illicit_rate(val)={vy.mean():.4f}{marker}  ({time.time()-t0:.1f}s)")

print(f"\nBest temporal-val F1: {best_val_f1:.4f} at epoch {best_epoch}")
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})


using class_weight=[1.0, 3.0]  (3.0 = even more moderate than phase-2 attention's 5.0)

Training 30 epochs, best epoch by TEMPORAL val F1@0.5 ...
epoch  1  train_loss=0.4610  val_F1@0.5=0.5638  illicit_rate(val)=0.1682  *  (0.5s)
epoch  2  train_loss=0.2091  val_F1@0.5=0.7609  illicit_rate(val)=0.1682  *  (0.2s)
epoch  3  train_loss=0.1589  val_F1@0.5=0.7872  illicit_rate(val)=0.1682  *  (0.2s)
epoch  4  train_loss=0.1402  val_F1@0.5=0.7970  illicit_rate(val)=0.1682  *  (0.2s)
epoch  5  train_loss=0.1269  val_F1@0.5=0.8388  illicit_rate(val)=0.1682  *  (0.2s)
epoch  6  train_loss=0.1181  val_F1@0.5=0.8303  illicit_rate(val)=0.1682  (0.2s)
epoch  7  train_loss=0.1103  val_F1@0.5=0.8173  illicit_rate(val)=0.1682  (0.2s)
epoch  8  train_loss=0.1059  val_F1@0.5=0.7958  illicit_rate(val)=0.1682  (0.2s)
epoch  9  train_loss=0.1001  val_F1@0.5=0.8531  illicit_rate(val)=0.1682  *  (0.2s)
epoch 10  train_loss=0.0944  val_F1@0.5=0.8521  illicit_rate(val)=0.1682  (0.2s)
epoch 11  train_loss=0.097

In [15]:
# ── Neural test eval ──────────────────────────────────────────────
ty, tp = run_eval(test_loader)
y_test = tx_label[test_idx_arr]
test_t_arr = tx_time[test_idx_arr]

# Threshold calibrated on the TEMPORAL val (closer to test distribution than random val)
vy, vp = run_eval(val_loader)
def best_threshold_f1(y, p):
    if not len(y) or len(np.unique(y)) < 2: return 0.5, 0.0
    prec, rec, thr = precision_recall_curve(y, p)
    f1s = 2 * prec * rec / (prec + rec + 1e-10)
    best = int(np.nanargmax(f1s[:-1]))
    return float(thr[best]), float(f1s[best])
thr_temporal_val, _ = best_threshold_f1(vy, vp)
print(f"Threshold calibrated on temporal val: {thr_temporal_val:.4f}  (val illicit_rate={vy.mean():.4f})")

f1_05_neural   = f1_score(y_test, (tp >= 0.5).astype(np.int64),               pos_label=1, zero_division=0)
f1_cal_neural  = f1_score(y_test, (tp >= thr_temporal_val).astype(np.int64),  pos_label=1, zero_division=0)
auc_neural     = roc_auc_score(y_test, tp)
prauc_neural   = average_precision_score(y_test, tp)
print("\n" + "=" * 70)
print("Phase 2 (simplified MLP) — test results")
print("=" * 70)
print(f"  F1@0.5={f1_05_neural:.4f}  F1@temporal-val-thr={f1_cal_neural:.4f}  "
      f"AUC={auc_neural:.4f}  PR-AUC={prauc_neural:.4f}")
y_pred_neural = (tp >= 0.5).astype(np.int64)
print(classification_report(y_test, y_pred_neural,
                            target_names=["licit","illicit"], digits=4, zero_division=0))

# ── RF baselines (trained on full t<=34, same as phase 1) ─────────
rf_train_mask = labelled & (tx_time <= TRAIN_END_FOR_TEST)
y_train_rf = tx_label[rf_train_mask]

print("=" * 70); print("RF baselines (re-run, same data as phase 1)"); print("=" * 70)
print("--- RF [raw 182] ---")
rf_raw = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                n_jobs=-1, random_state=RANDOM_SEED)
rf_raw.fit(tx_X_raw[rf_train_mask], y_train_rf)
yp_raw  = rf_raw.predict(tx_X_raw[test_mask])
ypr_raw = rf_raw.predict_proba(tx_X_raw[test_mask])[:, 1]
f1_raw   = f1_score(y_test, yp_raw, pos_label=1, zero_division=0)
auc_raw  = roc_auc_score(y_test, ypr_raw)
prauc_raw= average_precision_score(y_test, ypr_raw)
print(f"  F1@0.5={f1_raw:.4f}  AUC={auc_raw:.4f}  PR-AUC={prauc_raw:.4f}")

print("--- RF [raw + 103 traj] ---")
X_aug_train = np.concatenate([tx_X_raw[rf_train_mask], traj_X[rf_train_mask]], axis=1)
X_aug_test  = np.concatenate([tx_X_raw[test_mask],     traj_X[test_mask]],     axis=1)
rf_aug = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                n_jobs=-1, random_state=RANDOM_SEED)
rf_aug.fit(X_aug_train, y_train_rf)
yp_aug  = rf_aug.predict(X_aug_test)
ypr_aug = rf_aug.predict_proba(X_aug_test)[:, 1]
f1_aug   = f1_score(y_test, yp_aug, pos_label=1, zero_division=0)
auc_aug  = roc_auc_score(y_test, ypr_aug)
prauc_aug= average_precision_score(y_test, ypr_aug)
print(f"  F1@0.5={f1_aug:.4f}  AUC={auc_aug:.4f}  PR-AUC={prauc_aug:.4f}")

# ── Per-timestep test breakdown ───────────────────────────────────
print("\n" + "=" * 70); print("Per-timestep test F1 (illicit class)"); print("=" * 70)
print(f"{'t':>3}  {'n':>5}  {'illicit':>7}  {'RF[raw]':>8}  {'RF[aug]':>8}  {'MLP@0.5':>8}  {'MLP@cal':>8}")
for t in range(TRAIN_END_FOR_TEST + 1, N_TIME_STEPS + 1):
    m = (test_t_arr == t)
    if not m.any(): continue
    yt = y_test[m]
    if yt.sum() == 0:
        print(f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}  {'NaN':>8}  {'NaN':>8}  {'NaN':>8}  {'NaN':>8}")
        continue
    f1_r  = f1_score(yt, yp_raw[m],                              pos_label=1, zero_division=0)
    f1_a  = f1_score(yt, yp_aug[m],                              pos_label=1, zero_division=0)
    f1_n5 = f1_score(yt, (tp[m] >= 0.5).astype(int),             pos_label=1, zero_division=0)
    f1_nc = f1_score(yt, (tp[m] >= thr_temporal_val).astype(int),pos_label=1, zero_division=0)
    print(f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}  "
          f"{f1_r:>8.4f}  {f1_a:>8.4f}  {f1_n5:>8.4f}  {f1_nc:>8.4f}")

# ── Save numbers for cell 8 ───────────────────────────────────────
results = {
    "RF[raw 182]":            {"f1": f1_raw,       "auc": auc_raw,    "prauc": prauc_raw},
    "RF[raw + 103 traj]":     {"f1": f1_aug,       "auc": auc_aug,    "prauc": prauc_aug},
    "MLP @0.5":               {"f1": f1_05_neural, "auc": auc_neural, "prauc": prauc_neural},
    "MLP @temporal-val-thr":  {"f1": f1_cal_neural,"auc": auc_neural, "prauc": prauc_neural},
}
print(f"\nSaved neural test predictions in `tp` (and `vp` for val) — available for ensembling with RF.")


Threshold calibrated on temporal val: 0.4506  (val illicit_rate=0.1682)

Phase 2 (simplified MLP) — test results
  F1@0.5=0.4034  F1@temporal-val-thr=0.4219  AUC=0.8943  PR-AUC=0.4270
              precision    recall  f1-score   support

       licit     0.9525    0.9900    0.9709     15587
     illicit     0.6674    0.2890    0.4034      1083

    accuracy                         0.9445     16670
   macro avg     0.8099    0.6395    0.6871     16670
weighted avg     0.9340    0.9445    0.9340     16670

RF baselines (re-run, same data as phase 1)
--- RF [raw 182] ---
  F1@0.5=0.8044  AUC=0.9269  PR-AUC=0.7927
--- RF [raw + 103 traj] ---
  F1@0.5=0.8050  AUC=0.9369  PR-AUC=0.8101

Per-timestep test F1 (illicit class)
  t      n  illicit   RF[raw]   RF[aug]   MLP@0.5   MLP@cal
 35   1341      182    0.9805    0.9805    0.8359    0.8554
 36   1708       33    0.8814    0.9000    0.5667    0.5231
 37    498       40    0.7500    0.7692    0.4615    0.4848
 38    756      111    0.9429   

In [16]:
print("=" * 70)
print(f"{'Model':30s}  {'F1':>8s}  {'AUC':>8s}  {'PR-AUC':>8s}")
print("=" * 70)
for name, r in sorted(results.items(), key=lambda kv: -kv[1]["f1"]):
    print(f"  {name:28s}  {r['f1']:>8.4f}  {r['auc']:>8.4f}  {r['prauc']:>8.4f}")

best_neural_f1 = max(results['MLP @0.5']['f1'], results['MLP @temporal-val-thr']['f1'])
delta_vs_aug   = best_neural_f1 - results['RF[raw + 103 traj]']['f1']
print(f"\nBest neural F1: {best_neural_f1:.4f}  vs RF[aug] {results['RF[raw + 103 traj]']['f1']:.4f}  "
      f"(Δ={delta_vs_aug:+.4f})")

if delta_vs_aug >= -0.005:
    print("  ✅ Simplified neural matches/beats RF[aug] — viable consumer of trajectory features.")
elif delta_vs_aug >= -0.020:
    print("  ⚠️  Within 2 F1 of RF[aug] — close enough to ensemble (next step).")
else:
    print("  ❌ More than 2 F1 below RF[aug] — neural not competitive on this dataset.")
    print("     Honest framing: RF + traj features remain the strongest single model;")
    print("     neural is an ablation. Could still ensemble for marginal post-cliff gain.")

print("\nNotes:")
print(f"  - Architecture: 2-layer MLP, hidden={HIDDEN_DIM}, dropout={DROPOUT}, "
      f"{sum(p.numel() for p in model.parameters()):,} params (vs phase-2 attention's 45K).")
print(f"  - Train: t<= {TRAIN_END}, val: [{VAL_START}, {VAL_END}] (TEMPORAL), test: t > {TRAIN_END_FOR_TEST}.")
print(f"  - Same per-tx feature set as RF[aug]: 182 raw + 103 phase-1 trajectory = {F_IN}.")
print(f"  - Threshold calibrated on temporal val ({thr_temporal_val:.4f}).")
print(f"\nFor ensembling next step: `tp` (test) and `vp` (val) hold neural illicit probabilities;")
print(f"  feed them to RF as an extra column to get RF[raw + traj + neural_prob].")


Model                                 F1       AUC    PR-AUC
  RF[raw + 103 traj]              0.8050    0.9369    0.8101
  RF[raw 182]                     0.8044    0.9269    0.7927
  MLP @temporal-val-thr           0.4219    0.8943    0.4270
  MLP @0.5                        0.4034    0.8943    0.4270

Best neural F1: 0.4219  vs RF[aug] 0.8050  (Δ=-0.3831)
  ❌ More than 2 F1 below RF[aug] — neural not competitive on this dataset.
     Honest framing: RF + traj features remain the strongest single model;
     neural is an ablation. Could still ensemble for marginal post-cliff gain.

Notes:
  - Architecture: 2-layer MLP, hidden=32, dropout=0.3, 10,274 params (vs phase-2 attention's 45K).
  - Train: t<= 29, val: [30, 34] (TEMPORAL), test: t > 34.
  - Same per-tx feature set as RF[aug]: 182 raw + 103 phase-1 trajectory = 285.
  - Threshold calibrated on temporal val (0.4506).

For ensembling next step: `tp` (test) and `vp` (val) hold neural illicit probabilities;
  feed them to RF as an 